# 7.4 — Classification Metrics and the Accuracy Trap

**Chapter 7, section 7.3.2.** The starting point for **Exercise 3**.

**The question this notebook answers:** what does a hyperparameter search do when it is told to
maximize the wrong number?

The chapter's claim is that a classifier which always predicts *not fraud* scores 95 % accuracy
on a dataset that is 95 % legitimate, and that a search optimizing accuracy will therefore
choose it. This notebook builds that dataset, scores that classifier, then runs the search twice
over one grid — once told to maximize accuracy and once told to maximize the F-measure on the
fraud class — and reports the two different winners.

It closes with the one place in this chapter where the RDD-based `pyspark.mllib` API is the right
tool, because no DataFrame evaluator returns precision, recall, the F-measure and the confusion
matrix together.

Everything here is synthesized in the notebook from fixed seeds. No data file is needed.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-7.4")
         .master("local[4]")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)

Spark 4.2.0


---

## 1. A 95 / 5 dataset

Twenty thousand transactions, five percent of them fraudulent. Three predictors, all weakly
informative: the amount, the hour of the day, and whether the card was used abroad. Fraudulent
transactions tend to be larger and are much more often foreign, but neither signal separates the
classes cleanly — which is the realistic case and the one where the metric matters.

In [2]:
import pyspark.sql.functions as F

n = 20_000
tx = (spark.range(n)
      .withColumn("u", F.rand(seed=7))
      .withColumn("label", (F.col("u") < 0.05).cast("int"))
      .withColumn("amount", F.round(F.exp(F.randn(seed=8) + 3)
                                    + F.col("label") * F.rand(seed=9) * 120, 2))
      .withColumn("hour", (F.rand(seed=10) * 24).cast("int"))
      .withColumn("foreign", (F.rand(seed=11) < 0.05 + F.col("label") * 0.45).cast("int"))
      .drop("u", "id"))

tx.groupBy("label").count().orderBy("label").show()
tx.show(5)

+-----+-----+
|label|count|
+-----+-----+
|    0|19015|
|    1|  985|
+-----+-----+

+-----+------+----+-------+
|label|amount|hour|foreign|
+-----+------+----+-------+
|    0| 17.79|   4|      1|
|    0|113.83|  19|      0|
|    0|  4.15|  13|      0|
|    0| 12.95|  22|      0|
|    0| 89.66|   5|      0|
+-----+------+----+-------+
only showing top 5 rows


In [3]:
base_rate = tx.filter("label = 1").count() / tx.count()
print(f"fraud rate: {base_rate:.4f}")
print(f"a classifier that always says 'not fraud' is right {1 - base_rate:.2%} of the time")

fraud rate: 0.0493


a classifier that always says 'not fraud' is right 95.08% of the time


In [4]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=["amount", "hour", "foreign"], outputCol="features")
data = assembler.transform(tx).cache()

train, test = data.randomSplit([0.7, 0.3], seed=1)
print("training:", train.count(), " test:", test.count())

training: 13920  test: 6080


---

## 2. The useless classifier, scored

No model is fitted here. The prediction column is a literal zero, which is exactly what "always
predict *not fraud*" means.

In [5]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

constant = test.withColumn("prediction", F.lit(0.0))

accuracy = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
f1_fraud = MulticlassClassificationEvaluator(labelCol="label",
                                             metricName="fMeasureByLabel", metricLabel=1.0)
recall_fraud = MulticlassClassificationEvaluator(labelCol="label",
                                                 metricName="recallByLabel", metricLabel=1.0)

print(f"accuracy          {accuracy.evaluate(constant):.4f}")
print(f"recall on fraud   {recall_fraud.evaluate(constant):.4f}")
print(f"F-measure, fraud  {f1_fraud.evaluate(constant):.4f}")

accuracy          0.9526
recall on fraud   0.0000


F-measure, fraud  0.0000


Ninety-five percent accurate, and it has never once identified a fraudulent transaction. The two
metrics below it are zero, and they are zero for the same reason the first one is high.

---

## 3. One grid, two searches, two winners

A logistic regression is fitted, and the parameter searched is its decision `threshold` — the
probability above which a transaction is called fraud. Lowering it trades false positives for
caught fraud, which is precisely the trade the metric is supposed to arbitrate.

Both searches use the **same estimator**, the **same grid** and the **same three folds**. Only
the evaluator differs.

In [6]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

lr = LogisticRegression(labelCol="label", maxIter=30)

grid = ParamGridBuilder().addGrid(lr.threshold, [0.5, 0.2, 0.05, 0.02]).build()
print("parameter maps in the grid:", len(grid))
print("model fits per search:", len(grid), "×", 3, "folds =", len(grid) * 3, "plus one refit")

parameter maps in the grid: 4
model fits per search: 4 × 3 folds = 12 plus one refit


In [7]:
results = {}
for name, evaluator in (("accuracy", accuracy), ("F-measure on fraud", f1_fraud)):
    cv = CrossValidator(estimator=lr, evaluator=evaluator,
                        estimatorParamMaps=grid, numFolds=3, seed=1).fit(train)
    results[name] = cv
    scores = ", ".join(f"{t}: {s:.4f}"
                       for t, s in zip([0.5, 0.2, 0.05, 0.02], cv.avgMetrics))
    print(f"{name:<20} {scores}")
    print(f"{'':<20} winner: threshold = {cv.bestModel.getThreshold()}")
    print()

accuracy             0.5: 0.9535, 0.2: 0.9220, 0.05: 0.8878, 0.02: 0.6306
                     winner: threshold = 0.5



F-measure on fraud   0.5: 0.2584, 0.2: 0.3914, 0.05: 0.3744, 0.02: 0.2076
                     winner: threshold = 0.2



The search optimizing **accuracy** chose the highest threshold in the grid — the setting that
predicts fraud least often. The search optimizing the **F-measure on the fraud class** chose a
threshold four times lower.

Neither search is broken. Both faithfully maximized what they were handed. The first one was
handed a number that a constant classifier already scores 95 % on, so it walked towards the
constant classifier, which is where that number is highest.

---

## 4. What the two winners actually do

In [8]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

aupr = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR")
precision_fraud = MulticlassClassificationEvaluator(labelCol="label",
                                                    metricName="precisionByLabel",
                                                    metricLabel=1.0)

header = f"{'chosen by':<22}{'thresh':>8}{'accuracy':>10}{'precision':>11}{'recall':>9}{'F':>8}"
print(header)
print("-" * len(header))
for name, cv in results.items():
    p = cv.bestModel.transform(test)
    print(f"{name:<22}{cv.bestModel.getThreshold():>8}"
          f"{accuracy.evaluate(p):>10.4f}{precision_fraud.evaluate(p):>11.4f}"
          f"{recall_fraud.evaluate(p):>9.4f}{f1_fraud.evaluate(p):>8.4f}")

chosen by               thresh  accuracy  precision   recall       F
--------------------------------------------------------------------
accuracy                   0.5    0.9544     0.5873   0.1285  0.2108


F-measure on fraud         0.2    0.9237     0.3120   0.5069  0.3862


The accuracy-selected model is three percentage points more accurate and finds a small fraction
of the fraud. The F-selected model gives up those three points and finds several times as much.
Which of the two is wanted is a question about the cost of a missed fraud against the cost of a
false alarm, and no metric answers it on its own — but *accuracy* answers it wrongly by default
whenever one class is rare.

---

## 5. Precision, recall, F and the confusion matrix, together

Each DataFrame evaluator above returned one number, so the table in section 4 cost five separate
evaluations. The legacy RDD-based class `MulticlassMetrics` returns all of them from one object,
including the confusion matrix, which no DataFrame evaluator returns at all.

This is the one deliberate use of `pyspark.mllib` in the chapter. It is not dated code awaiting
modernization: the RDD-based API is in maintenance mode rather than deprecated, and it remains
the right tool where it still offers something the newer interface does not.

In [9]:
import warnings

from pyspark.mllib.evaluation import MulticlassMetrics


def m_metrics_l(ml_model, test_data):
    predictions = ml_model.transform(test_data).cache()

    # MulticlassMetrics expects (prediction, label), in that order. See the cell below.
    predictionAndLabels = predictions.select("prediction", "label") \
        .rdd.map(lambda x: (float(x[0]), float(x[1]))).cache()

    with warnings.catch_warnings():          # the class builds a legacy SQLContext internally
        warnings.simplefilter("ignore", FutureWarning)
        metrics = MulticlassMetrics(predictionAndLabels)
        print(f"Precision = {metrics.precision(1.0):.4f} "
              f"Recall = {metrics.recall(1.0):.4f} "
              f"F1 Score = {metrics.fMeasure(1.0):.4f}")
        print("Confusion matrix \n", metrics.confusionMatrix().toArray().astype(int))

    predictionAndLabels.unpersist()
    predictions.unpersist()

In [10]:
for name, cv in results.items():
    print(f"--- model chosen by {name} ---")
    m_metrics_l(cv.bestModel, test)
    print()

--- model chosen by accuracy ---


Precision = 0.5873 Recall = 0.1285 F1 Score = 0.2108
Confusion matrix 
 [[5766   26]
 [ 251   37]]

--- model chosen by F-measure on fraud ---
Precision = 0.3120 Recall = 0.5069 F1 Score = 0.3862
Confusion matrix 
 [[5470  322]
 [ 142  146]]



The matrix is read row by actual class and column by predicted class. Row 0 is the legitimate
transactions and row 1 the fraudulent ones, so the bottom-left cell is the fraud the model
missed and the top-right cell is the legitimate transactions it flagged. Every number in
section 4 is a ratio of two of these four cells, which is why one object can produce them all
and why the matrix is worth having rather than the ratios alone.

The two tables agree, which is the check that matters here.

In [11]:
# Same numbers, from the DataFrame evaluators of section 4.
for name, cv in results.items():
    p = cv.bestModel.transform(test)
    print(f"{name:<22} precision {precision_fraud.evaluate(p):.4f} "
          f"recall {recall_fraud.evaluate(p):.4f} F {f1_fraud.evaluate(p):.4f}")

accuracy               precision 0.5873 recall 0.1285 F 0.2108


F-measure on fraud     precision 0.3120 recall 0.5069 F 0.3862


### The argument order is not symmetric

`MulticlassMetrics` takes an RDD of **`(prediction, label)`** pairs, in that order. Passing
`(label, prediction)` raises nothing: the class computes happily and returns **precision and
recall exchanged, and the confusion matrix transposed**.

On an imbalanced problem the two numbers are far apart, so the mistake is loud once you look for
it — precision 0.31 against recall 0.51 becomes precision 0.51 against recall 0.31 — but nothing
in the output says which way round it is. The cell below runs it both ways so the failure is
visible rather than described.

In [12]:
correct = (results["F-measure on fraud"].bestModel.transform(test)
           .select("prediction", "label").rdd.map(lambda x: (float(x[0]), float(x[1]))))
swapped = (results["F-measure on fraud"].bestModel.transform(test)
           .select("label", "prediction").rdd.map(lambda x: (float(x[0]), float(x[1]))))

with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    for label, rdd in (("(prediction, label)  correct", correct),
                       ("(label, prediction)  wrong  ", swapped)):
        m = MulticlassMetrics(rdd)
        print(f"{label}  precision {m.precision(1.0):.4f}  recall {m.recall(1.0):.4f}")
        print("   ", m.confusionMatrix().toArray().astype(int).tolist())

(prediction, label)  correct  precision 0.3120  recall 0.5069
    [[5470, 322], [142, 146]]


(label, prediction)  wrong    precision 0.5069  recall 0.3120
    [[5470, 142], [322, 146]]


In [13]:
# The pairs the helper actually builds. Rows where the model said "fraud" are the interesting
# ones, so the sample is taken from those rather than from the top of the frame.
flagged = (results["F-measure on fraud"].bestModel.transform(test)
           .filter("prediction = 1").select("prediction", "label"))
print(flagged.rdd.map(lambda x: (float(x[0]), float(x[1]))).take(6))

[(1.0, 0.0), (1.0, 0.0), (1.0, 0.0), (1.0, 0.0), (1.0, 0.0), (1.0, 0.0)]


---

## 6. Exercise 3's arithmetic

The exercise uses a larger and more skewed dataset than this one: 200,000 transactions of which
1,400 are fraudulent.

In [14]:
total, fraud = 200_000, 1_400
print(f"fraud rate                      {fraud / total:.4%}")
print(f"accuracy of 'always legitimate' {(total - fraud) / total:.4%}")
print(f"recall on fraud                 {0 / fraud:.4f}")
print(f"precision on fraud              undefined — no positive prediction to be right about")

fraud rate                      0.7000%
accuracy of 'always legitimate' 99.3000%
recall on fraud                 0.0000
precision on fraud              undefined — no positive prediction to be right about


Accuracy rises as the class becomes rarer, which is the property that makes it useless here: the
metric improves as the problem gets harder.

---

## What to take away

A classifier that never predicts the rare class scores the base rate as its accuracy — **95 %**
on this dataset, **99.3 %** on Exercise 3's. A cross-validated search told to maximize accuracy
selects towards that classifier, and does so correctly, because it was asked to.

The fix is the evaluator, chosen before the search rather than after it. Where the whole picture
is wanted rather than one number, `pyspark.mllib.evaluation.MulticlassMetrics` returns precision,
recall, the F-measure and the confusion matrix from one object.

In [15]:
spark.stop()